In [ ]:
import os
import sys
import torch
import numpy as np
import json
import gc
from torchvision import transforms
from PIL import Image

# Project paths
PROJECT_ROOT = os.path.abspath(os.getcwd())
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
VIS_DIR = os.path.join(PROJECT_ROOT, "visualizations")

# Create directories
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)

# Add src to path
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

# Import custom modules
from src.model import get_model
from src.utils import calculate_dice, calculate_iou
from src.dataset import PanNukeDataset

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"📱 Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
print("\n🔧 Loading model...")
best_model_path = os.path.join(CHECKPOINT_DIR, "best_model.pth")

if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location='cpu')
    model = get_model(num_classes=2, pretrain=True)
    
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'], strict=False)
        best_dice = checkpoint.get('dice', 0)
    else:
        model.load_state_dict(checkpoint, strict=False)
        best_dice = 0
    
    model.eval()
    model = model.to(device)
    
    print(f"✅ Model loaded successfully!")
    print(f"   Best validation Dice: {best_dice:.4f}")
else:
    print(f"❌ Model not found at {best_model_path}")

In [ ]:
print("\n🔍 Loading validation dataset...")

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])

val_ds = PanNukeDataset(
    root_dir=DATA_DIR,
    split='validate',
    transform=transform,
    binary_mode=True
)

print(f"✅ Loaded {len(val_ds)} validation samples")

In [ ]:
print("\n🔬 Running inference on validation samples...")

NUM_SAMPLES = min(20, len(val_ds))
metrics = []

for i in range(NUM_SAMPLES):
    try:
        img, mask = val_ds[i]
        img_tensor = img.unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(img_tensor)['out']
            pred = torch.argmax(output, dim=1).cpu().squeeze()
        
        dice = calculate_dice(output.cpu(), mask.unsqueeze(0), num_classes=2)
        iou = calculate_iou(output.cpu(), mask.unsqueeze(0), num_classes=2)
        
        pred_nuclei = (pred == 1).sum().item()
        true_nuclei = (mask == 1).sum().item()
        
        intersection = ((pred == 1) & (mask == 1)).sum().item()
        precision = intersection / pred_nuclei if pred_nuclei > 0 else 0
        recall = intersection / true_nuclei if true_nuclei > 0 else 0
        
        metrics.append({
            'sample': i,
            'dice': dice,
            'iou': iou,
            'precision': precision,
            'recall': recall,
            'pred_nuclei': pred_nuclei,
            'true_nuclei': true_nuclei
        })
        
        if (i + 1) % 10 == 0:
            print(f"   Processed {i+1}/{NUM_SAMPLES} samples")
            
    except Exception as e:
        print(f"   Error on sample {i}: {e}")

print(f"\n✅ Inference complete on {len(metrics)} samples")

In [ ]:
print("\n" + "="*70)
print("RESULTS SUMMARY")
print("="*70)

if len(metrics) > 0:
    # Calculate statistics
    dice_scores = [m['dice'] for m in metrics]
    iou_scores = [m['iou'] for m in metrics]
    precision_scores = [m['precision'] for m in metrics]
    recall_scores = [m['recall'] for m in metrics]
    
    mean_dice = np.mean(dice_scores)
    std_dice = np.std(dice_scores)
    mean_iou = np.mean(iou_scores)
    
    print(f"\n🎯 OVERALL PERFORMANCE:")
    print(f"   Mean Dice: {mean_dice:.4f} ± {std_dice:.4f}")
    print(f"   Mean IoU: {mean_iou:.4f} ± {np.std(iou_scores):.4f}")
    print(f"   Mean Precision: {np.mean(precision_scores):.4f}")
    print(f"   Mean Recall: {np.mean(recall_scores):.4f}")
    
    print(f"\n📊 PER-SAMPLE RESULTS:")
    for m in metrics:
        print(f"   Sample {m['sample']}: Dice={m['dice']:.4f}, IoU={m['iou']:.4f}, "
              f"Pred={m['pred_nuclei']}, True={m['true_nuclei']}")
    
    # Save metrics
    import pandas as pd
    df = pd.DataFrame(metrics)
    df.to_csv(os.path.join(RESULTS_DIR, "demo_metrics.csv"), index=False)
    print(f"\n✅ Metrics saved to: {os.path.join(RESULTS_DIR, 'demo_metrics.csv')}")
    
else:
    print("⚠️ No metrics calculated - check if validation dataset loaded correctly")

In [ ]:
import os
# Cell 6: Save Raw Data Only (No Matplotlib)
print("\n" + "="*70)
print("💾 SAVING RAW PREDICTION DATA")
print("="*70)

import numpy as np
from PIL import Image
import gc

# Create directories
RAW_DIR = os.path.join(PROJECT_ROOT, "raw_predictions")
os.makedirs(RAW_DIR, exist_ok=True)

# Save first 3 samples as numpy arrays and raw images
num_save = min(3, len(val_ds))
saved_info = []

for idx in range(num_save):
    try:
        print(f"\n   Processing sample {idx}...")
        
        # Get sample
        img, mask = val_ds[idx]
        
        # Run prediction
        img_tensor = img.unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(img_tensor)['out']
            pred = torch.argmax(output, dim=1).cpu().squeeze()
        
        # Calculate Dice
        dice = calculate_dice(output.cpu(), mask.unsqueeze(0), num_classes=2)
        
        # Denormalize image for saving (manual denormalization)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_np = img.cpu().numpy().transpose(1, 2, 0)
        img_denorm = std * img_np + mean
        img_denorm = np.clip(img_denorm * 255, 0, 255).astype(np.uint8)
        
        # Save as numpy arrays
        np.save(os.path.join(RAW_DIR, f"sample_{idx}_image.npy"), img_denorm)
        np.save(os.path.join(RAW_DIR, f"sample_{idx}_mask.npy"), mask.numpy())
        np.save(os.path.join(RAW_DIR, f"sample_{idx}_pred.npy"), pred.numpy())
        
        # Save as PNG images using PIL (no matplotlib)
        Image.fromarray(img_denorm).save(os.path.join(RAW_DIR, f"sample_{idx}_image.png"))
        Image.fromarray((mask.numpy() * 255).astype(np.uint8)).save(os.path.join(RAW_DIR, f"sample_{idx}_mask.png"))
        Image.fromarray((pred.numpy() * 255).astype(np.uint8)).save(os.path.join(RAW_DIR, f"sample_{idx}_pred.png"))
        
        # Save metadata
        metadata = {
            'sample': idx,
            'dice': float(dice),
            'pred_nuclei': int((pred == 1).sum().item()),
            'true_nuclei': int((mask == 1).sum().item()),
            'image_shape': list(img_denorm.shape),
            'mask_shape': list(mask.shape)
        }
        
        with open(os.path.join(RAW_DIR, f"sample_{idx}_metadata.json"), 'w') as f:
            json.dump(metadata, f, indent=2)
        
        saved_info.append(metadata)
        print(f"      ✅ Dice: {dice:.4f}")
        print(f"      ✅ Saved to: {RAW_DIR}/sample_{idx}_*.png")
        
        # Clear memory
        del img, mask, img_tensor, output, pred, img_denorm
        gc.collect()
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"      ❌ Error: {e}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n" + "="*70)
print("✅ RAW DATA SAVED SUCCESSFULLY!")
print("="*70)
print(f"\n📁 Location: {RAW_DIR}/")
print("\nSaved files per sample:")
print("   • sample_X_image.png - Original image (viewable)")
print("   • sample_X_mask.png - Ground truth mask")
print("   • sample_X_pred.png - Model prediction")
print("   • sample_X_*.npy - NumPy arrays")
print("   • sample_X_metadata.json - Metrics")

print(f"\n📊 Results:")
for info in saved_info:
    print(f"   Sample {info['sample']}: Dice={info['dice']:.4f}, "
          f"Pred={info['pred_nuclei']}, True={info['true_nuclei']}")

print(f"\n💡 To view images, open folder: {RAW_DIR}")
print(f"   Or run: explorer {RAW_DIR} (Windows) or open {RAW_DIR} (Mac)")
print("\n" + "="*70)

In [ ]:
# Cell: Final Status Summary
print("\n" + "="*70)
print("🎉 DEMO COMPLETED SUCCESSFULLY!")
print("="*70)

import json
import os

RAW_DIR = os.path.join(PROJECT_ROOT, "raw_predictions")

# Load metrics from saved metadata
if len(metrics) > 0:
    mean_dice = np.mean([m['dice'] for m in metrics])
    std_dice = np.std([m['dice'] for m in metrics])
    mean_iou = np.mean([m['iou'] for m in metrics])
    
    print(f"""
✅ MODEL PERFORMANCE:
   • Model: DeepLabV3 with ResNet50
   • Best Validation Dice: {best_dice:.4f} ({best_dice*100:.2f}%)
   • Tested on: {len(metrics)} validation samples
   • Mean Dice: {mean_dice:.4f} ± {std_dice:.4f}
   • Mean IoU: {mean_iou:.4f} ± {np.std([m['iou'] for m in metrics]):.4f}
""")

# Check saved visualizations
if os.path.exists(RAW_DIR):
    png_files = [f for f in os.listdir(RAW_DIR) if f.endswith('.png')]
    if png_files:
        print(f"📁 VISUALIZATIONS SAVED:")
        print(f"   Location: {RAW_DIR}/")
        print(f"   Files: {len(png_files)} PNG images")
        print(f"\n   Sample files:")
        for f in sorted(png_files)[:6]:
            print(f"      • {f}")
        if len(png_files) > 6:
            print(f"      ... and {len(png_files)-6} more")

print(f"""
📁 GENERATED FILES:
   • Model: {CHECKPOINT_DIR}/best_model.pth
   • Metrics: {RESULTS_DIR}/demo_metrics.csv
   • Visualizations: {RAW_DIR}/ (PNG images and numpy arrays)
   • Metadata: {RAW_DIR}/sample_*_metadata.json

💡 NEXT STEPS:
   1. View images in: {RAW_DIR}
   2. Use model for inference on new images
   3. Export to ONNX for production: torch.onnx.export(model, dummy_input, "model.onnx")
""")

print("="*70)
print("✅ DEMO NOTEBOOK COMPLETE!")
print("="*70)

Create Comparison Images (Ground Truth vs Prediction)

In [ ]:
# Cell: Create Comparison Images (Ground Truth vs Prediction)
print("\n" + "="*70)
print("🖼️ CREATING COMPARISON IMAGES")
print("="*70)

import numpy as np
from PIL import Image, ImageDraw, ImageFont
import os
import gc

# Create comparison directory
COMPARE_DIR = os.path.join(PROJECT_ROOT, "comparison_images")
os.makedirs(COMPARE_DIR, exist_ok=True)

def create_comparison_image(img_array, mask_array, pred_array, dice_score, sample_idx):
    """
    Create a side-by-side comparison image with:
    - Original image
    - Ground truth mask (green overlay)
    - Prediction mask (red overlay)
    - Combined overlay
    """
    # Denormalize image first (convert from tensor to RGB)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    
    # If img_array is still normalized (values between -2 and 2)
    if img_array.min() < 0:
        img_array = std * img_array + mean
        img_array = np.clip(img_array, 0, 1)
    
    # Convert to 0-255 range
    img_rgb = (img_array * 255).astype(np.uint8)
    
    # Create RGB versions of masks
    # Ground truth: Green overlay
    gt_overlay = np.zeros_like(img_rgb)
    gt_overlay[..., 1] = (mask_array == 1).astype(np.uint8) * 200  # Green channel
    
    # Prediction: Red overlay
    pred_overlay = np.zeros_like(img_rgb)
    pred_overlay[..., 0] = (pred_array == 1).astype(np.uint8) * 200  # Red channel
    
    # Combined: Green (GT) + Red (Prediction)
    combined = img_rgb.copy()
    # Add ground truth (green)
    combined[..., 1] = np.maximum(combined[..., 1], gt_overlay[..., 1])
    # Add prediction (red)
    combined[..., 0] = np.maximum(combined[..., 0], pred_overlay[..., 0])
    
    # Create side-by-side comparison using PIL
    # Image 1: Original
    img_pil = Image.fromarray(img_rgb)
    
    # Image 2: Ground Truth Overlay
    gt_pil = Image.fromarray(np.clip(img_rgb + gt_overlay, 0, 255).astype(np.uint8))
    
    # Image 3: Prediction Overlay
    pred_pil = Image.fromarray(np.clip(img_rgb + pred_overlay, 0, 255).astype(np.uint8))
    
    # Image 4: Combined (GT Green + Prediction Red)
    combined_pil = Image.fromarray(combined)
    
    # Create a single wide image with all 4
    width, height = img_pil.size
    combined_width = width * 4
    combined_height = height + 60  # Add space for text
    
    # Create new image
    result = Image.new('RGB', (combined_width, combined_height), color='white')
    
    # Paste images
    result.paste(img_pil, (0, 0))
    result.paste(gt_pil, (width, 0))
    result.paste(pred_pil, (width * 2, 0))
    result.paste(combined_pil, (width * 3, 0))
    
    # Add text labels
    draw = ImageDraw.Draw(result)
    
    # Try to use a font, fallback to default
    try:
        font = ImageFont.truetype("arial.ttf", 16)
    except:
        font = ImageFont.load_default()
    
    # Labels
    draw.text((width//2 - 40, height + 10), "Original", fill='black', font=font)
    draw.text((width + width//2 - 60, height + 10), "Ground Truth", fill='black', font=font)
    draw.text((width*2 + width//2 - 50, height + 10), "Prediction", fill='black', font=font)
    draw.text((width*3 + width//2 - 40, height + 10), "Combined", fill='black', font=font)
    
    # Add Dice score
    draw.text((10, height + 35), f"Dice: {dice_score:.3f}", fill='black', font=font)
    
    return result

# Process first 3 samples
num_compare = min(3, len(val_ds))
print(f"\n📸 Creating comparison images for {num_compare} samples...")

for idx in range(num_compare):
    try:
        print(f"\n   Sample {idx}:")
        
        # Get sample
        img, mask = val_ds[idx]
        
        # Run prediction
        img_tensor = img.unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(img_tensor)['out']
            pred = torch.argmax(output, dim=1).cpu().squeeze()
        
        # Calculate Dice
        dice = calculate_dice(output.cpu(), mask.unsqueeze(0), num_classes=2)
        
        # Denormalize image
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_np = img.cpu().numpy().transpose(1, 2, 0)
        img_denorm = std * img_np + mean
        img_denorm = np.clip(img_denorm, 0, 1)
        
        # Create comparison image
        comparison = create_comparison_image(
            img_denorm, 
            mask.numpy(), 
            pred.numpy(), 
            dice, 
            idx
        )
        
        # Save
        save_path = os.path.join(COMPARE_DIR, f"comparison_sample_{idx}_dice_{dice:.3f}.png")
        comparison.save(save_path)
        
        print(f"      ✅ Dice: {dice:.4f}")
        print(f"      ✅ Saved: {os.path.basename(save_path)}")
        
        # Clear memory
        del img, mask, img_tensor, output, pred, img_denorm, comparison
        gc.collect()
        
    except Exception as e:
        print(f"      ❌ Error: {e}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n" + "="*70)
print("✅ COMPARISON IMAGES CREATED!")
print("="*70)
print(f"\n📁 Location: {COMPARE_DIR}/")
print("\nWhat each image shows (4 panels):")
print("   1. Original Image")
print("   2. Ground Truth (Green overlay)")
print("   3. Prediction (Red overlay)")
print("   4. Combined (GT Green + Pred Red)")
print("\n   • Green = Correctly identified nuclei")
print("   • Red = False positives (predicted but not in GT)")
print("   • Yellow/Orange = True positives (correct predictions)")

print(f"\n💡 To view images:")
if os.name == 'nt':
    print(f'   !explorer "{COMPARE_DIR}"')
else:
    print(f'   !open "{COMPARE_DIR}"')

print("\n" + "="*70)

In [ ]:
# Cell: Create Comparison Images (Run This!)
print("\n" + "="*70)
print("🖼️ CREATING COMPARISON IMAGES")
print("="*70)

import numpy as np
from PIL import Image, ImageDraw, ImageFont
import os
import gc

# Create directories
COMPARE_DIR = os.path.join(PROJECT_ROOT, "comparison_images")
OVERLAY_DIR = os.path.join(PROJECT_ROOT, "overlay_comparisons")
RAW_DIR = os.path.join(PROJECT_ROOT, "raw_predictions")

os.makedirs(COMPARE_DIR, exist_ok=True)
os.makedirs(OVERLAY_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)

print(f"✅ Directories created:")
print(f"   • {COMPARE_DIR}")
print(f"   • {OVERLAY_DIR}")
print(f"   • {RAW_DIR}")

# Check if val_ds exists
if 'val_ds' not in dir():
    print("\n⚠️ Loading validation dataset...")
    from torchvision import transforms
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                            std=[0.229, 0.224, 0.225])
    ])
    val_ds = PanNukeDataset(
        root_dir=DATA_DIR,
        split='validate',
        transform=transform,
        binary_mode=True
    )
    print(f"✅ Loaded {len(val_ds)} samples")

def denormalize_image(img_tensor):
    """Convert normalized tensor to RGB image"""
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_np = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img_denorm = std * img_np + mean
    img_denorm = np.clip(img_denorm, 0, 1)
    return (img_denorm * 255).astype(np.uint8)

# Process samples
num_samples = min(3, len(val_ds))
print(f"\n📸 Creating images for {num_samples} samples...")

for idx in range(num_samples):
    try:
        print(f"\n   Sample {idx}:")
        
        # Get sample
        img, mask = val_ds[idx]
        
        # Run prediction
        img_tensor = img.unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(img_tensor)['out']
            pred = torch.argmax(output, dim=1).cpu().squeeze()
        
        # Calculate Dice
        dice = calculate_dice(output.cpu(), mask.unsqueeze(0), num_classes=2)
        print(f"      Dice score: {dice:.4f}")
        
        # Convert to RGB images
        img_rgb = denormalize_image(img)
        mask_bool = (mask.numpy() == 1)
        pred_bool = (pred.numpy() == 1)
        
        # --- 1. Save individual images ---
        Image.fromarray(img_rgb).save(os.path.join(RAW_DIR, f"sample_{idx}_image.png"))
        Image.fromarray((mask_bool * 255).astype(np.uint8)).save(os.path.join(RAW_DIR, f"sample_{idx}_mask.png"))
        Image.fromarray((pred_bool * 255).astype(np.uint8)).save(os.path.join(RAW_DIR, f"sample_{idx}_pred.png"))
        print(f"      ✅ Saved individual images to raw_predictions/")
        
        # --- 2. Create simple overlay (2-panel) ---
        # Ground Truth overlay (green)
        gt_overlay = img_rgb.copy()
        gt_overlay[mask_bool, 1] = 255  # Green channel
        gt_overlay[mask_bool, 0] = gt_overlay[mask_bool, 0] // 2
        gt_overlay[mask_bool, 2] = gt_overlay[mask_bool, 2] // 2
        
        # Prediction overlay (red)
        pred_overlay = img_rgb.copy()
        pred_overlay[pred_bool, 0] = 255  # Red channel
        pred_overlay[pred_bool, 1] = pred_overlay[pred_bool, 1] // 2
        pred_overlay[pred_bool, 2] = pred_overlay[pred_bool, 2] // 2
        
        # Create side-by-side
        width, height = img_rgb.shape[1], img_rgb.shape[0]
        combined = Image.new('RGB', (width * 2, height + 40), color='white')
        combined.paste(Image.fromarray(gt_overlay), (0, 0))
        combined.paste(Image.fromarray(pred_overlay), (width, 0))
        
        # Add text
        draw = ImageDraw.Draw(combined)
        try:
            font = ImageFont.truetype("arial.ttf", 14)
        except:
            font = ImageFont.load_default()
        
        draw.text((width//2 - 60, height + 10), "Ground Truth (Green)", fill='black', font=font)
        draw.text((width + width//2 - 50, height + 10), "Prediction (Red)", fill='black', font=font)
        draw.text((10, height + 25), f"Dice: {dice:.3f}", fill='black', font=font)
        
        combined.save(os.path.join(OVERLAY_DIR, f"overlay_sample_{idx}_dice_{dice:.3f}.png"))
        print(f"      ✅ Saved overlay to overlay_comparisons/")
        
        # --- 3. Create 4-panel comparison ---
        # Create 4 panels
        panel_width = width
        panel_height = height
        result = Image.new('RGB', (panel_width * 4, panel_height + 50), color='white')
        
        # Panel 1: Original
        result.paste(Image.fromarray(img_rgb), (0, 0))
        
        # Panel 2: Ground Truth (green overlay)
        result.paste(Image.fromarray(gt_overlay), (panel_width, 0))
        
        # Panel 3: Prediction (red overlay)
        result.paste(Image.fromarray(pred_overlay), (panel_width * 2, 0))
        
        # Panel 4: Combined (GT + Prediction)
        combined_overlay = img_rgb.copy()
        combined_overlay[mask_bool, 1] = 255  # Green for GT
        combined_overlay[pred_bool, 0] = 255   # Red for Pred
        result.paste(Image.fromarray(combined_overlay), (panel_width * 3, 0))
        
        # Add labels
        draw = ImageDraw.Draw(result)
        labels = ["Original", "Ground Truth", "Prediction", "Combined"]
        for i, label in enumerate(labels):
            x = panel_width * i + panel_width//2 - 40
            draw.text((x, panel_height + 10), label, fill='black', font=font)
        
        draw.text((10, panel_height + 30), f"Dice: {dice:.3f}", fill='black', font=font)
        
        result.save(os.path.join(COMPARE_DIR, f"comparison_sample_{idx}_dice_{dice:.3f}.png"))
        print(f"      ✅ Saved comparison to comparison_images/")
        
        # Clear memory
        del img, mask, img_tensor, output, pred
        gc.collect()
        
    except Exception as e:
        print(f"      ❌ Error: {e}")
        import traceback
        traceback.print_exc()
        continue

print("\n" + "="*70)
print("✅ ALL IMAGES CREATED SUCCESSFULLY!")
print("="*70)
print(f"\n📁 Files saved in:")
print(f"   • {COMPARE_DIR}/ - 4-panel comparisons")
print(f"   • {OVERLAY_DIR}/ - Simple overlays (GT vs Pred)")
print(f"   • {RAW_DIR}/ - Individual images")

# List created files
print("\n📸 Created files:")
for dir_name, dir_path in [("Comparison", COMPARE_DIR), ("Overlay", OVERLAY_DIR), ("Raw", RAW_DIR)]:
    if os.path.exists(dir_path):
        files = [f for f in os.listdir(dir_path) if f.endswith('.png')]
        if files:
            print(f"\n   {dir_name} images ({len(files)} files):")
            for f in sorted(files)[:5]:
                print(f"      • {f}")
            if len(files) > 5:
                print(f"      ... and {len(files)-5} more")

print("\n💡 To view images, open the folders in file explorer")
print("="*70)

In [ ]:
# Cell: Open Folder with All Visualizations
print("\n📁 Opening folder with all visualizations...")

import subprocess
import platform

# Open the main visualizations folder
VISUALIZATIONS_DIR = PROJECT_ROOT

print(f"\n📂 Visualizations saved in:")
print(f"   • {os.path.join(PROJECT_ROOT, 'comparison_images')} - Side-by-side comparisons")
print(f"   • {os.path.join(PROJECT_ROOT, 'overlay_comparisons')} - Simple overlays")
print(f"   • {os.path.join(PROJECT_ROOT, 'raw_predictions')} - Individual images")

# Optionally open the folder
open_folder = input("\nOpen folder to view images? (y/n): ")

if open_folder.lower() == 'y':
    if platform.system() == 'Windows':
        os.startfile(PROJECT_ROOT)
    elif platform.system() == 'Darwin':
        subprocess.run(['open', PROJECT_ROOT])
    else:
        subprocess.run(['xdg-open', PROJECT_ROOT])
    print("✅ Folder opened!")

In [ ]:
# Cell: Display Generated Images in Notebook
print("\n" + "="*70)
print("📸 DISPLAYING IMAGES IN NOTEBOOK")
print("="*70)

from IPython.display import Image as IPImage, display
import os

# Check if images exist
COMPARE_DIR = os.path.join(PROJECT_ROOT, "comparison_images")

if os.path.exists(COMPARE_DIR):
    # Get list of comparison images
    comparison_images = sorted([f for f in os.listdir(COMPARE_DIR) if f.endswith('.png')])
    
    if comparison_images:
        print(f"\n✅ Found {len(comparison_images)} comparison images\n")
        
        # Display first 4 images (safe)
        for img_file in comparison_images[:4]:
            img_path = os.path.join(COMPARE_DIR, img_file)
            print(f"📄 {img_file}")
            display(IPImage(filename=img_path, width=800))
            print("-" * 70)
        
        if len(comparison_images) > 4:
            print(f"\n💡 {len(comparison_images)-2} more images saved to: {COMPARE_DIR}")
    else:
        print("⚠️ No images found. Run the image creation cell first.")
else:
    print(f"⚠️ Directory not found: {COMPARE_DIR}")
    print("   Run the image creation cell first")